# AirDrawVocab — Train & Demo trên Google Colab

Notebook tổng hợp: **tải dữ liệu QuickDraw → train model nhận diện hình vẽ (19 lớp) → đánh giá → lưu model → demo nhận diện ngay trên Colab** (tải ảnh lên hoặc vẽ trực tiếp bằng chuột).

### Cách dùng nhanh
1. Vào menu **Runtime → Change runtime type → Hardware accelerator = GPU (T4)** rồi **Save**.
2. Chạy lần lượt từ trên xuống, hoặc **Runtime → Run all**.
3. Toàn bộ chỉ mất vài phút nhờ GPU + mixed precision + batch lớn.

### Vì sao notebook này train nhanh & ổn định
- **GPU + mixed precision (float16)** và **batch lớn** → mỗi epoch chỉ vài giây.
- **Kiến trúc CNN không dùng BatchNormalization** (BatchNorm gây lỗi val/inference ~ngẫu nhiên trên một số bản TensorFlow/Keras 3) → kết quả ổn định, đáng tin cậy.
- **EarlyStopping** tự dừng khi đã hội tụ, không train thừa.

19 lớp: `apple, baseball, book, bowtie, diamond, dog, door, envelope, eye, fish, hat, leaf, lightning, moon, pants, scissors, square, star, t-shirt`.

## 1. Kiểm tra GPU

In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus)
if not gpus:
    print("\n[!] CHƯA bật GPU. Vào Runtime → Change runtime type → GPU (T4) rồi chạy lại.")
    print("    (Vẫn train được trên CPU nhưng chậm hơn nhiều.)")
else:
    print("\nĐã có GPU — train sẽ rất nhanh.")

## 2. Cấu hình & import

Bạn có thể chỉnh `PER_CLASS` (số mẫu mỗi lớp). Nhiều dữ liệu hơn → chính xác hơn nhưng tải/ train lâu hơn một chút. Trên GPU, `10000` là cân bằng tốt.

In [ ]:
import os
import io
import json
from io import BytesIO
from pathlib import Path

import numpy as np
import requests
import cv2
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision

# ---- 19 lớp (đúng thứ tự, khớp với app/backend) ----
CATEGORIES = [
    "apple", "baseball", "book", "bowtie", "diamond",
    "dog", "door", "envelope", "eye", "fish",
    "hat", "leaf", "lightning", "moon", "pants",
    "scissors", "square", "star", "t-shirt",
]
NUM_CLASSES = len(CATEGORIES)

# ---- Cấu hình train (chỉnh ở đây) ----
PER_CLASS   = 10000   # số mẫu mỗi lớp tải về
TRAIN_PC    = 8000
VAL_PC      = 1000
TEST_PC     = 1000
BATCH_SIZE  = 1024    # batch lớn cho GPU; nếu hết RAM/CPU hãy giảm còn 256
EPOCHS      = 40
PATIENCE    = 6
LR          = 1.5e-3
SEED        = 42

DATA_DIR   = Path("data/npy_28"); DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = Path("models");      MODELS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODELS_DIR / "airdrawvocab_best_advanced.keras"
CATEGORIES_PATH = MODELS_DIR / "categories.json"

# ---- Mixed precision: nhanh hơn nhiều trên GPU ----
USE_MIXED = bool(tf.config.list_physical_devices("GPU"))
if USE_MIXED:
    mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision: BẬT (mixed_float16)")
else:
    print("Mixed precision: TẮT (không có GPU)")

# ---- Nghĩa tiếng Việt + câu ví dụ (cho phần demo/chatbot) ----
VI_MEANINGS = {
    "apple":"quả táo","baseball":"bóng chày","book":"quyển sách","bowtie":"nơ bướm",
    "diamond":"kim cương","dog":"con chó","door":"cánh cửa","envelope":"phong bì",
    "eye":"con mắt","fish":"con cá","hat":"cái mũ","leaf":"chiếc lá","lightning":"tia sét",
    "moon":"mặt trăng","pants":"quần dài","scissors":"cái kéo","square":"hình vuông",
    "star":"ngôi sao","t-shirt":"áo thun",
}
EXAMPLE_SENTENCES = {
    "apple":"I eat an apple every morning.","baseball":"He plays baseball after school.",
    "book":"This book is very interesting.","bowtie":"He wears a bowtie at the party.",
    "diamond":"The diamond is very shiny.","dog":"The dog is friendly.","door":"Please close the door.",
    "envelope":"She puts the letter in an envelope.","eye":"My eye is blue.","fish":"The fish swims in the water.",
    "hat":"I wear a hat on sunny days.","leaf":"A leaf falls from the tree.","lightning":"Lightning appears during the storm.",
    "moon":"The moon is bright tonight.","pants":"These pants are black.","scissors":"I cut paper with scissors.",
    "square":"This is a red square.","star":"A star shines in the sky.","t-shirt":"I like this t-shirt.",
}
print("Số lớp:", NUM_CLASSES)

## 3. Tải dữ liệu QuickDraw (tải từng phần — rất nhanh)

Chỉ tải đủ số mẫu cần dùng cho mỗi lớp (dùng HTTP Range), không tải full file hàng trăm MB.

In [ ]:
QD_URL = "https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/{}.npy"

def download_partial(label, n):
    """Tải ~n mẫu đầu của 1 lớp QuickDraw (numpy_bitmap, 28x28 uint8)."""
    out = DATA_DIR / f"{label}.npy"
    if out.exists() and len(np.load(out, mmap_mode="r")) >= n:
        return np.load(out)
    nbytes = 256 + n * 784
    r = requests.get(QD_URL.format(label), headers={"Range": f"bytes=0-{nbytes}"}, timeout=180)
    r.raise_for_status()
    raw = r.content
    assert raw[:6] == b"\x93NUMPY", f"{label}: file không hợp lệ"
    major = raw[6]
    if major == 1:
        data_off = 10 + int.from_bytes(raw[8:10], "little")
    else:
        data_off = 12 + int.from_bytes(raw[8:12], "little")
    avail = (len(raw) - data_off) // 784
    arr = np.frombuffer(raw[data_off:data_off + avail * 784], dtype=np.uint8).reshape(avail, 784)
    np.save(out, arr)
    return arr

for i, label in enumerate(CATEGORIES, 1):
    arr = download_partial(label, PER_CLASS)
    print(f"[{i:2d}/{NUM_CLASSES}] {label:10s}: {len(arr)} mẫu")
print("Xong tải dữ liệu vào", DATA_DIR)

## 4. Nạp & chia dữ liệu (train / val / test)

In [ ]:
def load_split(train_pc, val_pc, test_pc, seed=SEED):
    need = train_pc + val_pc + test_pc
    rng = np.random.default_rng(seed)
    Xtr, Ytr, Xva, Yva, Xte, Yte = [], [], [], [], [], []
    for cid, cat in enumerate(CATEGORIES):
        data = np.load(DATA_DIR / f"{cat}.npy")
        data = data[data.sum(axis=1) > 0]            # bỏ mẫu rỗng
        if len(data) < need:
            raise ValueError(f"{cat}: chỉ có {len(data)} mẫu, cần {need}. Hãy tăng PER_CLASS hoặc giảm split.")
        idx = rng.permutation(len(data))[:need]
        data = (data[idx].astype("float32") / 255.0).reshape(-1, 28, 28, 1)
        Xtr.append(data[:train_pc]);                         Ytr.append(np.full(train_pc, cid))
        Xva.append(data[train_pc:train_pc+val_pc]);          Yva.append(np.full(val_pc, cid))
        Xte.append(data[train_pc+val_pc:need]);              Yte.append(np.full(test_pc, cid))
    return (np.concatenate(Xtr), np.concatenate(Ytr),
            np.concatenate(Xva), np.concatenate(Yva),
            np.concatenate(Xte), np.concatenate(Yte))

x_tr, y_tr, x_va, y_va, x_te, y_te = load_split(TRAIN_PC, VAL_PC, TEST_PC)
print(f"train={len(x_tr)}  val={len(x_va)}  test={len(x_te)}")

## 5. Xây model (CNN không BatchNorm) + augmentation trong tf.data

In [ ]:
def build_model(dropout=0.4):
    inputs = keras.Input(shape=(28, 28, 1), name="drawing")
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x); x = layers.Dropout(0.25)(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x); x = layers.Dropout(0.25)(x)
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x); x = layers.Dropout(0.3)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    # dtype="float32" ở output để ổn định số học khi dùng mixed precision
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32", name="predictions")(x)
    return keras.Model(inputs, outputs, name="airdraw_cnn")


def augment(x, y):
    # dịch nhẹ ±3px bằng pad + random crop (không lật vì nhiều vật có hướng)
    pad = 3
    xp = tf.pad(x, [[0, 0], [pad, pad], [pad, pad], [0, 0]])
    xp = tf.image.random_crop(xp, tf.shape(x))
    return xp, y


def make_ds(x, y, batch, training):
    y_cat = keras.utils.to_categorical(y, NUM_CLASSES)
    ds = tf.data.Dataset.from_tensor_slices((x, y_cat))
    if training:
        ds = ds.shuffle(len(x), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.batch(batch).map(augment, num_parallel_calls=tf.data.AUTOTUNE)
        return ds.prefetch(tf.data.AUTOTUNE)
    return ds.batch(batch).cache().prefetch(tf.data.AUTOTUNE)

model = build_model()
model.summary()

## 6. Train (GPU + mixed precision + early stopping)

In [ ]:
import time

model.compile(
    optimizer=keras.optimizers.Adam(LR),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max",
                                  patience=PATIENCE, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                      patience=2, min_lr=1e-6, verbose=1),
]

t0 = time.time()
history = model.fit(
    make_ds(x_tr, y_tr, BATCH_SIZE, True),
    validation_data=make_ds(x_va, y_va, BATCH_SIZE, False),
    epochs=EPOCHS, callbacks=callbacks, verbose=2,
)
print(f"\nTổng thời gian train: {time.time()-t0:.1f}s")

## 7. Đánh giá: accuracy, top-3, per-class, confusion matrix & đường học

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

prob = model.predict(make_ds(x_te, y_te, BATCH_SIZE, False), verbose=0)
pred = prob.argmax(1)
acc = (pred == y_te).mean()
top3 = np.mean([t in row for t, row in zip(y_te, np.argsort(prob, 1)[:, -3:])])
print(f"TEST accuracy = {acc*100:.2f}%   top-3 = {top3*100:.2f}%   ({len(y_te)} mẫu)\n")
print(classification_report(y_te, pred, target_names=CATEGORIES, digits=3))

# Confusion matrix
cm = confusion_matrix(y_te, pred)
plt.figure(figsize=(10, 8))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xticks(range(NUM_CLASSES), CATEGORIES, rotation=45, ha="right")
plt.yticks(range(NUM_CLASSES), CATEGORIES)
plt.xlabel("Dự đoán"); plt.ylabel("Thực tế"); plt.title(f"Confusion Matrix (acc={acc*100:.1f}%)")
plt.tight_layout(); plt.show()

# Đường học
h = history.history
plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1); plt.plot(h["accuracy"], label="train"); plt.plot(h["val_accuracy"], label="val")
plt.title("Accuracy"); plt.xlabel("epoch"); plt.legend(); plt.grid(alpha=.3)
plt.subplot(1, 2, 2); plt.plot(h["loss"], label="train"); plt.plot(h["val_loss"], label="val")
plt.title("Loss"); plt.xlabel("epoch"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 8. Lưu model + categories.json

Lưu vào `models/`. Bỏ comment phần cuối nếu muốn **tải file về máy** hoặc **lưu vào Google Drive** để dùng cho web/desktop app.

In [ ]:
model.save(MODEL_PATH)
CATEGORIES_PATH.write_text(json.dumps(CATEGORIES, ensure_ascii=False), encoding="utf-8")
print("Đã lưu:", MODEL_PATH)
print("Đã lưu:", CATEGORIES_PATH)

# --- Tải file về máy (bỏ comment để dùng) ---
# from google.colab import files
# files.download(str(MODEL_PATH))
# files.download(str(CATEGORIES_PATH))

# --- Hoặc lưu vào Google Drive (bỏ comment để dùng) ---
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil
# dst = "/content/drive/MyDrive/AirDrawVocab_models"
# os.makedirs(dst, exist_ok=True)
# shutil.copy(MODEL_PATH, dst); shutil.copy(CATEGORIES_PATH, dst)
# print("Đã copy model vào", dst)

## 9. Tiền xử lý ảnh + hàm dự đoán (giống hệt backend)

In [ ]:
def preprocess_image(image_bytes):
    """Ảnh canvas/upload -> (1,28,28,1): nền đen, nét trắng, căn giữa, vuông."""
    image = Image.open(BytesIO(image_bytes)).convert("RGBA")
    rgba = np.array(image)
    rgb = rgba[:, :, :3].astype(np.float32)
    alpha = rgba[:, :, 3:4].astype(np.float32) / 255.0
    white = np.full_like(rgb, 255.0)
    rgb = (rgb * alpha + white * (1 - alpha)).astype(np.uint8)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    if float(gray.mean()) > 127:        # nền sáng -> đảo thành nền đen nét trắng
        gray = 255 - gray
    _, thresh = cv2.threshold(gray, 20, 255, cv2.THRESH_BINARY)
    coords = cv2.findNonZero(thresh)
    if coords is not None:
        x, y, w, h = cv2.boundingRect(coords)
        pad = max(8, int(max(w, h) * 0.20))
        x1, y1 = max(0, x - pad), max(0, y - pad)
        x2, y2 = min(gray.shape[1], x + w + pad), min(gray.shape[0], y + h + pad)
        gray = gray[y1:y2, x1:x2]
    h, w = gray.shape[:2]
    side = max(h, w, 1)
    square = np.zeros((side, side), dtype=np.uint8)
    yo, xo = (side - h) // 2, (side - w) // 2
    square[yo:yo + h, xo:xo + w] = gray
    resized = cv2.resize(square, (28, 28), interpolation=cv2.INTER_AREA)
    return (resized.astype("float32") / 255.0)[None, ..., None]


def predict_bytes(image_bytes, topk=3):
    x = preprocess_image(image_bytes)
    p = model.predict(x, verbose=0)[0]
    order = p.argsort()[::-1][:topk]
    return [(CATEGORIES[int(i)], float(p[int(i)])) for i in order]


def chatbot_reply(top3):
    label, conf = top3[0]
    vi = VI_MEANINGS.get(label, label)
    ex = EXAMPLE_SENTENCES.get(label, f"This is a {label}.")
    pct = round(conf * 100, 1)
    others = ", ".join(l for l, _ in top3)
    if conf >= 0.8:
        return f"Đây là **{label}** ({vi}) — độ tin cậy {pct}%. Ví dụ: {ex}"
    if conf >= 0.5:
        return f"Mình đoán **{label}** ({vi}), {pct}%. Ví dụ: {ex}\nCác khả năng gần: {others}. Hãy vẽ to & rõ hơn nhé."
    return f"Chưa chắc lắm — gần nhất là **{label}** ({vi}), {pct}%. Top: {others}. Thử vẽ lại to, ít nét thừa hơn."

print("Sẵn sàng dự đoán.")

## 10. Demo A — Tải ảnh hình vẽ lên để nhận diện
Chạy cell, bấm **Choose Files**, chọn 1 ảnh hình vẽ (nền sáng nét đậm là tốt nhất).

In [ ]:
from google.colab import files
import matplotlib.pyplot as plt

uploaded = files.upload()
for name, content in uploaded.items():
    top3 = predict_bytes(content)
    plt.figure(figsize=(3, 3))
    plt.imshow(preprocess_image(content)[0, :, :, 0], cmap="gray")
    plt.title(f"{top3[0][0]} ({top3[0][1]*100:.1f}%)"); plt.axis("off"); plt.show()
    print("File:", name)
    print("Top-3:", ", ".join(f"{l} {c*100:.1f}%" for l, c in top3))
    print(chatbot_reply(top3), "\n")

## 11. Demo B — Vẽ trực tiếp bằng chuột trên Colab

Chạy cell → một bảng vẽ hiện ra. Vẽ bằng chuột, bấm **Nhận diện** (hoặc **Xóa** để vẽ lại). Có thể chạy lại cell nhiều lần để thử nhiều hình.

In [ ]:
from IPython.display import Javascript, display
from google.colab.output import eval_js
from base64 import b64decode
import matplotlib.pyplot as plt

def draw_pad():
    js = Javascript('''
    async function drawPad() {
      const div = document.createElement('div');
      const canvas = document.createElement('canvas');
      canvas.width = 280; canvas.height = 280;
      canvas.style.border = '2px solid #333'; canvas.style.background = '#fff';
      canvas.style.cursor = 'crosshair'; canvas.style.touchAction = 'none';
      const br = document.createElement('br');
      const btn = document.createElement('button'); btn.textContent = '✅ Nhận diện';
      const clr = document.createElement('button'); clr.textContent = '🗑️ Xóa';
      btn.style.margin = clr.style.margin = '6px';
      div.appendChild(canvas); div.appendChild(br); div.appendChild(btn); div.appendChild(clr);
      document.body.appendChild(div);

      const ctx = canvas.getContext('2d');
      ctx.fillStyle = '#fff'; ctx.fillRect(0, 0, 280, 280);
      ctx.lineWidth = 12; ctx.lineCap = 'round'; ctx.lineJoin = 'round'; ctx.strokeStyle = '#000';
      let drawing = false;
      const pos = e => { const r = canvas.getBoundingClientRect();
        return [e.clientX - r.left, e.clientY - r.top]; };
      canvas.addEventListener('mousedown', e => { drawing = true; const [x,y]=pos(e); ctx.beginPath(); ctx.moveTo(x,y); });
      canvas.addEventListener('mousemove', e => { if(drawing){ const [x,y]=pos(e); ctx.lineTo(x,y); ctx.stroke(); } });
      window.addEventListener('mouseup', () => drawing = false);
      clr.onclick = () => { ctx.fillStyle = '#fff'; ctx.fillRect(0, 0, 280, 280); };

      const data = await new Promise(res => { btn.onclick = () => res(canvas.toDataURL('image/png')); });
      div.remove();
      return data;
    }
    ''')
    display(js)
    data_url = eval_js('drawPad()')
    return b64decode(data_url.split(',')[1])

img_bytes = draw_pad()
top3 = predict_bytes(img_bytes)
plt.figure(figsize=(3, 3))
plt.imshow(preprocess_image(img_bytes)[0, :, :, 0], cmap="gray")
plt.title(f"{top3[0][0]} ({top3[0][1]*100:.1f}%)"); plt.axis("off"); plt.show()
print("Top-3:", ", ".join(f"{l} {c*100:.1f}%" for l, c in top3))
print(chatbot_reply(top3))

## Mẹo tăng tốc / tăng độ chính xác

- **Train nhanh nhất:** giảm `PER_CLASS` (vd 4000) và `EPOCHS` (vd 20). Trên GPU đã rất nhanh sẵn.
- **Chính xác hơn:** tăng `PER_CLASS` (15000–20000) và `EPOCHS` (50). QuickDraw còn rất nhiều dữ liệu.
- **Hết RAM khi batch lớn:** giảm `BATCH_SIZE` xuống 256.
- Model lưu ở `models/airdrawvocab_best_advanced.keras` + `models/categories.json` — copy 2 file này vào project local (thư mục `models/`) là web/desktop app dùng được ngay.

## 12. Demo C — Giao diện web ngay trong Colab (giống localhost) 🎮

Chạy cell dưới để hiện một **app web mini** ngay trong output: bảng vẽ + nút **Nhận diện**, panel kết quả (nhãn, độ tin cậy, top-3, chatbot, ảnh minh họa), và **chế độ game QuickDraw** (đếm giờ, mạng, điểm, streak, AI đoán liên tục).

Cách hoạt động: JavaScript trong Colab gọi ngược về Python (`google.colab.kernel.invokeFunction`) để chạy model `model` đã train ở trên — nên **phải chạy sau khi đã train/nạp model**.

> Lưu ý: vẽ bằng **chuột**. Phần webcam + vẽ bằng ngón tay (MediaPipe) và đăng nhập khuôn mặt chỉ có ở app desktop/web localhost, không chạy ổn định trong Colab nên không đưa vào đây.

In [ ]:
import base64
from PIL import ImageDraw

# ---------- Ảnh minh họa offline theo nhãn (port từ backend) ----------
def _encode_png_data_uri(image):
    buf = BytesIO(); image.save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode("ascii")

def create_reference_image(label, size=768):
    image = Image.new("RGB", (size, size), (246, 248, 252))
    overlay = Image.new("RGBA", (size, size), (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    cx, cy = size // 2, size // 2
    for r in range(size // 2, 0, -12):
        tone = int(250 - (size // 2 - r) * 0.025)
        draw.ellipse((cx - r, cy - r, cx + r, cy + r), fill=(tone, min(tone + 1, 255), 255, 6))
    if label == "apple":
        draw.ellipse((230, 220, 540, 595), fill=(178, 34, 42, 255), outline=(120, 20, 28, 255), width=4)
        draw.rectangle((370, 150, 395, 240), fill=(88, 52, 29, 255))
        draw.ellipse((395, 145, 510, 215), fill=(47, 129, 78, 255))
    elif label == "baseball":
        draw.ellipse((190, 170, 578, 558), fill=(250, 250, 246, 255), outline=(210, 210, 204, 255), width=5)
        for x in (300, 470):
            draw.arc((x - 90, 190, x + 90, 545), 250, 110, fill=(180, 28, 42, 255), width=7)
    elif label == "book":
        draw.rounded_rectangle((175, 210, 585, 545), radius=24, fill=(37, 99, 235, 255), outline=(20, 62, 150, 255), width=5)
        draw.rectangle((205, 235, 550, 520), fill=(239, 246, 255, 255))
        draw.line((382, 225, 382, 530), fill=(148, 163, 184, 255), width=5)
    elif label == "bowtie":
        draw.polygon([(110, 250), (340, 360), (110, 520)], fill=(20, 20, 28, 255))
        draw.polygon([(658, 250), (428, 360), (658, 520)], fill=(20, 20, 28, 255))
        draw.rounded_rectangle((335, 300, 435, 455), radius=24, fill=(45, 45, 58, 255))
    elif label == "diamond":
        draw.polygon([(384, 125), (600, 310), (384, 650), (168, 310)], fill=(210, 244, 255, 255), outline=(14, 116, 144, 255))
        draw.line((168, 310, 600, 310), fill=(14, 116, 144, 255), width=4)
    elif label == "dog":
        draw.ellipse((210, 190, 558, 580), fill=(180, 120, 72, 255), outline=(120, 74, 44, 255), width=5)
        draw.ellipse((155, 220, 290, 430), fill=(130, 78, 46, 255))
        draw.ellipse((478, 220, 613, 430), fill=(130, 78, 46, 255))
        draw.ellipse((285, 330, 315, 360), fill=(20, 20, 20, 255))
        draw.ellipse((453, 330, 483, 360), fill=(20, 20, 20, 255))
        draw.ellipse((355, 405, 410, 450), fill=(35, 20, 18, 255))
    elif label == "door":
        draw.rounded_rectangle((235, 105, 545, 665), radius=18, fill=(128, 78, 40, 255), outline=(78, 48, 28, 255), width=6)
        draw.ellipse((465, 380, 500, 415), fill=(229, 180, 70, 255))
    elif label == "envelope":
        draw.rounded_rectangle((145, 245, 623, 530), radius=10, fill=(255, 255, 255, 255), outline=(160, 160, 160, 255), width=4)
        draw.line((145, 245, 384, 395, 623, 245), fill=(160, 160, 160, 255), width=4)
    elif label == "eye":
        draw.ellipse((115, 265, 653, 505), fill=(255, 255, 255, 255), outline=(38, 38, 48, 255), width=7)
        draw.ellipse((300, 245, 468, 525), fill=(59, 130, 246, 255))
        draw.ellipse((347, 315, 421, 450), fill=(10, 10, 15, 255))
    elif label == "fish":
        draw.ellipse((145, 250, 555, 520), fill=(247, 156, 74, 255), outline=(170, 83, 30, 255), width=5)
        draw.polygon([(555, 385), (690, 250), (690, 520)], fill=(239, 120, 55, 255), outline=(170, 83, 30, 255))
        draw.ellipse((245, 330, 280, 365), fill=(20, 20, 20, 255))
    elif label == "hat":
        draw.pieslice((155, 210, 555, 590), 180, 360, fill=(30, 100, 210, 255), outline=(20, 64, 140, 255), width=5)
        draw.rounded_rectangle((135, 420, 660, 500), radius=38, fill=(37, 99, 235, 255))
    elif label == "leaf":
        draw.ellipse((220, 130, 555, 630), fill=(47, 150, 82, 255), outline=(20, 100, 55, 255), width=5)
        draw.line((385, 170, 385, 640), fill=(220, 255, 220, 210), width=5)
    elif label == "lightning":
        draw.polygon([(430, 80), (220, 410), (360, 410), (280, 690), (570, 330), (425, 330)], fill=(250, 204, 21, 255), outline=(180, 120, 0, 255))
    elif label == "moon":
        draw.ellipse((170, 120, 610, 560), fill=(225, 229, 235, 255), outline=(180, 185, 195, 255), width=5)
        draw.ellipse((260, 190, 330, 260), fill=(195, 200, 210, 170))
    elif label == "pants":
        draw.polygon([(245, 115), (535, 115), (610, 665), (445, 665), (390, 305), (335, 665), (170, 665)], fill=(37, 99, 235, 255), outline=(20, 64, 140, 255))
    elif label == "scissors":
        draw.line((230, 210, 560, 590), fill=(110, 116, 128, 255), width=16)
        draw.line((560, 210, 230, 590), fill=(110, 116, 128, 255), width=16)
        draw.ellipse((145, 150, 285, 290), outline=(37, 99, 235, 255), width=16)
        draw.ellipse((145, 520, 285, 660), outline=(37, 99, 235, 255), width=16)
    elif label == "square":
        draw.rectangle((200, 200, 568, 568), fill=(202, 138, 74, 255), outline=(120, 72, 36, 255), width=8)
    elif label == "star":
        pts = []
        for i in range(10):
            ang = -np.pi / 2 + i * np.pi / 5
            r = 265 if i % 2 == 0 else 110
            pts.append((cx + r * np.cos(ang), cy + r * np.sin(ang)))
        draw.polygon(pts, fill=(245, 184, 35, 255), outline=(170, 110, 20, 255))
    elif label == "t-shirt":
        draw.polygon([(250, 150), (330, 110), (384, 170), (438, 110), (518, 150), (610, 295), (520, 355), (520, 650), (248, 650), (248, 355), (158, 295)], fill=(255, 255, 255, 255), outline=(90, 96, 110, 255))
    else:
        draw.ellipse((180, 180, 588, 588), fill=(226, 232, 240, 255), outline=(100, 116, 139, 255), width=6)
    image = Image.alpha_composite(image.convert("RGBA"), overlay).convert("RGB")
    return _encode_png_data_uri(image)

print("Hàm ảnh minh họa sẵn sàng.")

In [ ]:
from IPython.display import HTML, JSON, display

# ---------- Callbacks Python để JS gọi ngược ----------
def _cb_predict(data_url, target=""):
    img_bytes = base64.b64decode(data_url.split(",", 1)[1])
    top3 = predict_bytes(img_bytes, topk=3)
    label, conf = top3[0]
    return JSON({
        "label": label,
        "meaning_vi": VI_MEANINGS.get(label, label),
        "confidence": conf,
        "confidence_percent": round(conf * 100, 2),
        "top3": [{"label": l, "meaning_vi": VI_MEANINGS.get(l, l), "confidence": c} for l, c in top3],
        "chatbot_reply": chatbot_reply(top3),
    })

_ref_cache = {}
def _cb_reference(label):
    if label not in _ref_cache:
        _ref_cache[label] = create_reference_image(label)
    return JSON({"label": label, "image": _ref_cache[label]})

try:
    from google.colab import output as _colab_output
    _colab_output.register_callback("notebook.predict", _cb_predict)
    _colab_output.register_callback("notebook.reference", _cb_reference)
    _ON_COLAB = True
except Exception as _e:
    _ON_COLAB = False
    print("[!] Không phải môi trường Colab nên UI tương tác có thể không gọi được model:", _e)

DRAWING_HINTS = {
    "apple": "Vẽ vòng tròn, thêm cuống và lá nhỏ phía trên.",
    "baseball": "Vẽ vòng tròn, thêm 2 đường cong khâu bóng hai bên.",
    "book": "Vẽ hình chữ nhật, kẻ gáy sách ở giữa.",
    "bowtie": "Vẽ 2 tam giác chạm nhau, thêm nút ở giữa.",
    "diamond": "Vẽ hình thoi: đỉnh trên, dưới và hai góc ngang.",
    "dog": "Vẽ đầu tròn, 2 tai, mắt và mũi.",
    "door": "Vẽ hình chữ nhật đứng, thêm tay nắm tròn.",
    "envelope": "Vẽ hình chữ nhật ngang, thêm nét chữ V.",
    "eye": "Vẽ oval nằm ngang, thêm tròng và con ngươi.",
    "fish": "Vẽ thân oval, đuôi tam giác và mắt.",
    "hat": "Vẽ nửa vòng tròn, thêm vành ngang.",
    "leaf": "Vẽ oval nhọn, thêm gân lá ở giữa.",
    "lightning": "Vẽ đường zigzag nhọn từ trên xuống.",
    "moon": "Vẽ trăng lưỡi liềm bằng 2 đường cong.",
    "pants": "Vẽ cạp quần và 2 ống quần.",
    "scissors": "Vẽ chữ X, thêm 2 vòng tròn tay cầm.",
    "square": "Vẽ 4 cạnh đều, khép kín.",
    "star": "Vẽ 5 đỉnh nhọn nối liền.",
    "t-shirt": "Vẽ thân áo, 2 tay áo và cổ áo.",
}

HTML_UI = r"""
<style>
  #adv-app{font-family:Segoe UI,Roboto,Arial,sans-serif;color:#0f172a;max-width:1000px;margin:auto}
  #adv-app .card{background:#fff;border:1px solid #e2e8f0;border-radius:14px;padding:16px;margin-bottom:14px;box-shadow:0 1px 3px rgba(15,23,42,.06)}
  #adv-app h2{margin:0 0 4px;font-size:20px}
  #adv-app h3{margin:0 0 8px;font-size:15px;color:#334155}
  #adv-app .muted{color:#64748b;font-size:13px}
  #adv-app .hud{display:flex;gap:8px;background:#13355f;border-radius:10px;padding:10px;color:#fff;flex-wrap:wrap}
  #adv-app .hud div{flex:1;min-width:70px;text-align:center}
  #adv-app .hud small{display:block;color:#9db7da;font-size:11px;letter-spacing:.5px}
  #adv-app .hud b{font-size:17px}
  #adv-app button{border:0;border-radius:8px;padding:8px 14px;font-weight:600;cursor:pointer;margin:3px}
  #adv-app .primary{background:#2563eb;color:#fff}
  #adv-app .ghost{background:#eef2f7;color:#1e293b}
  #adv-app .row{display:flex;gap:14px;flex-wrap:wrap}
  #adv-app .col{flex:1;min-width:300px}
  #adv-app canvas{border:2px dashed #cbd5e1;border-radius:10px;background:#fff;cursor:crosshair;touch-action:none}
  #adv-app .bar{height:10px;background:#e2e8f0;border-radius:6px;overflow:hidden;margin-top:3px}
  #adv-app .bar>span{display:block;height:100%;background:#2563eb}
  #adv-app .top3 div{margin-bottom:8px;font-size:13px}
  #adv-app .reply{background:#eff6ff;border:1px solid #dbeafe;border-radius:10px;padding:10px;margin-top:8px;font-size:14px;line-height:1.5}
  #adv-app img.ref{width:100%;max-width:240px;border-radius:10px;border:1px solid #e2e8f0;display:none}
  #adv-app .status{background:#f8fafc;border:1px solid #e2e8f0;border-radius:8px;padding:8px;margin-top:8px;font-size:13px}
</style>

<div id="adv-app">
  <div class="card">
    <h2>AirDrawVocab — nhận diện hình vẽ &amp; chatbot</h2>
    <div class="muted">Vẽ một hình như apple, dog, star, moon, book... rồi bấm Nhận diện. Hoặc bật Game để AI đoán liên tục như QuickDraw.</div>
  </div>

  <div class="card">
    <h3>🎮 Game QuickDraw</h3>
    <div class="hud">
      <div><small>DRAW</small><b id="g-target">---</b></div>
      <div><small>TIME</small><b id="g-time">20</b></div>
      <div><small>LEVEL</small><b id="g-level">1/6</b></div>
      <div><small>LIVES</small><b id="g-lives">♥♥♥</b></div>
      <div><small>SCORE</small><b id="g-score">0</b></div>
      <div><small>STREAK</small><b id="g-streak">0</b></div>
    </div>
    <div style="margin-top:8px">
      <button class="primary" id="g-start">Bắt đầu</button>
      <button class="ghost" id="g-skip" disabled>Bỏ qua</button>
      <button class="ghost" id="g-clear">Xóa nét</button>
    </div>
    <div class="status" id="g-status">Bấm Bắt đầu để chơi. Vẽ đúng từ trong DRAW trước khi hết giờ.</div>
    <div class="status" id="g-guess">AI chưa đoán.</div>
  </div>

  <div class="row">
    <div class="col card">
      <h3>Bảng vẽ</h3>
      <div class="muted">Vẽ nét đậm, to, ở giữa khung để kết quả tốt hơn.</div>
      <div style="margin:8px 0"><canvas id="adv-canvas" width="280" height="280"></canvas></div>
      <div>Độ dày nét: <input id="adv-brush" type="range" min="6" max="22" value="12"></div>
      <div style="margin-top:6px">
        <button class="ghost" id="adv-clear">Xóa</button>
        <button class="primary" id="adv-predict">Nhận diện</button>
      </div>
    </div>

    <div class="col card">
      <h3>Kết quả</h3>
      <div id="adv-empty" class="muted">Hãy vẽ một hình rồi bấm Nhận diện.</div>
      <div id="adv-result" style="display:none">
        <div style="font-size:20px;font-weight:700" id="adv-label">---</div>
        <div class="muted" id="adv-conf"></div>
        <div class="top3" id="adv-top3" style="margin-top:10px"></div>
        <div class="reply" id="adv-reply"></div>
        <img class="ref" id="adv-ref" alt="ảnh minh họa">
      </div>
    </div>
  </div>
</div>

<script>
(function(){
  const CATS = __CATEGORIES__;
  const HINTS = __HINTS__;
  const $ = id => document.getElementById(id);
  const canvas = $("adv-canvas"), ctx = canvas.getContext("2d");
  let hasDrawn=false, drawing=false, busy=false;

  function clearCanvas(){ ctx.fillStyle="#fff"; ctx.fillRect(0,0,280,280);
    ctx.lineCap="round"; ctx.lineJoin="round"; ctx.strokeStyle="#000"; hasDrawn=false; }
  clearCanvas();
  function brush(){ return parseInt($("adv-brush").value)||12; }
  const pos = e => { const r=canvas.getBoundingClientRect();
    const t=e.touches?e.touches[0]:e; return [t.clientX-r.left, t.clientY-r.top]; };
  function down(e){ e.preventDefault(); drawing=true; hasDrawn=true; const [x,y]=pos(e);
    ctx.lineWidth=brush(); ctx.beginPath(); ctx.moveTo(x,y); }
  function move(e){ if(!drawing)return; e.preventDefault(); const [x,y]=pos(e);
    ctx.lineWidth=brush(); ctx.lineTo(x,y); ctx.stroke(); }
  function up(){ drawing=false; }
  canvas.addEventListener("mousedown",down); canvas.addEventListener("mousemove",move);
  window.addEventListener("mouseup",up);
  canvas.addEventListener("touchstart",down,{passive:false});
  canvas.addEventListener("touchmove",move,{passive:false});
  canvas.addEventListener("touchend",up);

  async function callPredict(target){
    const res = await google.colab.kernel.invokeFunction("notebook.predict",[canvas.toDataURL("image/png"), target||""],{});
    return res.data["application/json"];
  }
  const refCache={};
  async function showRef(label){
    if(!refCache[label]){
      const res = await google.colab.kernel.invokeFunction("notebook.reference",[label],{});
      refCache[label]=res.data["application/json"].image;
    }
    const img=$("adv-ref"); img.src=refCache[label]; img.style.display="block";
  }

  function renderResult(r){
    $("adv-empty").style.display="none"; $("adv-result").style.display="block";
    $("adv-label").textContent = r.label + " (" + r.meaning_vi + ")";
    $("adv-conf").textContent = "Độ tin cậy: " + r.confidence_percent + "%";
    $("adv-top3").innerHTML = r.top3.map(it =>
      "<div><b>"+it.label+"</b> — "+it.meaning_vi+" · "+(it.confidence*100).toFixed(1)+"%"+
      "<div class='bar'><span style='width:"+(it.confidence*100).toFixed(0)+"%'></span></div></div>").join("");
    $("adv-reply").innerHTML = r.chatbot_reply.replace(/\*\*(.*?)\*\*/g,"<b>$1</b>");
    showRef(r.label);
  }

  $("adv-clear").onclick = ()=>{ clearCanvas(); $("adv-result").style.display="none"; $("adv-empty").style.display="block"; };
  $("adv-predict").onclick = async ()=>{
    if(!hasDrawn){ alert("Hãy vẽ một hình trước nhé."); return; }
    if(busy) return; busy=true; $("adv-predict").textContent="Đang nhận diện...";
    try{ renderResult(await callPredict("")); } catch(e){ alert("Lỗi: "+e); }
    finally{ busy=false; $("adv-predict").textContent="Nhận diện"; }
  };

  // -------- Game QuickDraw --------
  const G={active:false,round:false,level:1,maxLevel:6,lives:3,maxLives:3,score:0,streak:0,time:20,target:""};
  let tTimer=null, gTimer=null, gBusy=false;
  function hud(){
    $("g-target").textContent=G.target||"---";
    $("g-time").textContent=String(G.time).padStart(2,"0");
    $("g-level").textContent=G.level+"/"+G.maxLevel;
    $("g-lives").textContent="♥".repeat(Math.max(G.lives,0))+"♡".repeat(Math.max(G.maxLives-G.lives,0));
    $("g-score").textContent=G.score; $("g-streak").textContent=G.streak;
    $("g-skip").disabled=!G.round;
  }
  function stopTimers(){ if(tTimer)clearInterval(tTimer); if(gTimer)clearInterval(gTimer); tTimer=gTimer=null; }
  function pickTarget(){ const c=CATS.filter(x=>x!==G.target); return c[Math.floor(Math.random()*c.length)]; }
  function startRound(){
    clearCanvas(); $("adv-result").style.display="none"; $("adv-empty").style.display="block";
    G.target=pickTarget(); G.time=20; G.round=true; hud();
    $("g-status").textContent="Vẽ '"+G.target+"' trong 20 giây. Gợi ý: "+(HINTS[G.target]||"");
    $("g-guess").textContent="AI chưa đoán.";
    stopTimers();
    tTimer=setInterval(()=>{ if(!G.round)return; G.time--; hud(); if(G.time<=0) timeout(); },1000);
    gTimer=setInterval(guess,1300);
  }
  async function guess(){
    if(!hasDrawn||gBusy||!G.round) return; gBusy=true;
    try{
      const r=await callPredict(G.target);
      $("g-guess").innerHTML="AI đoán: <b>"+r.label+"</b> "+r.confidence_percent+"% · "+
        r.top3.map(t=>t.label+" "+(t.confidence*100).toFixed(0)+"%").join(", ");
      showRef(r.label);
      if(r.label===G.target && r.confidence>=0.45) success(r);
    }catch(e){} finally{ gBusy=false; }
  }
  function nextRound(delay){ setTimeout(()=>{ if(!G.active)return;
    if(G.level>=G.maxLevel||G.lives<=0){ over(); return; } G.level++; startRound(); }, delay||900); }
  function success(r){ if(!G.round)return; G.round=false; stopTimers();
    const bonus=100+G.time*5+G.streak*20; G.score+=bonus; G.streak++; hud();
    $("g-status").textContent="🎉 Chính xác! AI nhận ra "+r.label+". +"+bonus+" điểm."; nextRound(); }
  function timeout(){ if(!G.round)return; G.round=false; stopTimers(); G.lives--; G.streak=0; hud();
    $("g-status").textContent="⏰ Hết giờ! Từ cần vẽ là "+G.target+"."; nextRound(); }
  function over(){ stopTimers(); G.active=false; G.round=false; hud();
    $("g-start").textContent="Chơi lại";
    $("g-status").textContent="Kết thúc! Tổng điểm: "+G.score+"."; }
  $("g-start").onclick=()=>{ G.active=true; G.level=1; G.lives=G.maxLives; G.score=0; G.streak=0; G.target="";
    $("g-start").textContent="Chơi lại"; hud(); startRound(); };
  $("g-skip").onclick=()=>{ if(!G.round)return; G.round=false; stopTimers(); G.streak=0; hud();
    $("g-status").textContent="Đã bỏ qua. Thử từ khác."; nextRound(350); };
  $("g-clear").onclick=()=>{ clearCanvas(); $("g-guess").textContent="AI chưa đoán."; };
  hud();
})();
</script>
"""

html = (HTML_UI
        .replace("__CATEGORIES__", json.dumps(CATEGORIES))
        .replace("__HINTS__", json.dumps(DRAWING_HINTS, ensure_ascii=False)))
display(HTML(html))

## 13. Camera: bật/tắt + vẽ bằng ngón tay (MediaPipe) 📷

Giống "Camera Game Mode" trên web: **Bật camera** → đưa bàn tay vào khung, **giơ ngón trỏ** (gập các ngón khác) để vẽ nét, **xòe cả bàn tay** để xóa, rồi bấm **Nhận diện**.

> Yêu cầu: cho phép trình duyệt **truy cập camera** và có **internet** (tải MediaPipe Hands từ CDN). Phải chạy **phần 12** trước (để đăng ký callback model). Nếu trình duyệt chặn camera trong Colab, hãy bấm cho phép ở thanh địa chỉ.

In [ ]:
from IPython.display import HTML, display

# Đảm bảo callback đã đăng ký (phòng khi chạy cell này mà quên phần 12)
try:
    from google.colab import output as _co
    _co.register_callback("notebook.predict", _cb_predict)
    _co.register_callback("notebook.reference", _cb_reference)
except Exception as _e:
    print("[!] Không phải Colab hoặc chưa chạy phần 12:", _e)

CAM_UI = r"""
<style>
  #cam-app{font-family:Segoe UI,Roboto,Arial,sans-serif;color:#0f172a;max-width:900px;margin:auto}
  #cam-app .card{background:#fff;border:1px solid #e2e8f0;border-radius:14px;padding:16px;margin-bottom:14px}
  #cam-app button{border:0;border-radius:8px;padding:8px 14px;font-weight:600;cursor:pointer;margin:3px}
  #cam-app .primary{background:#2563eb;color:#fff}.ghost{background:#eef2f7;color:#1e293b}
  #cam-app .danger{background:#ef4444;color:#fff}
  #cam-app .stage{position:relative;width:480px;max-width:100%}
  #cam-app video{display:block;width:480px;max-width:100%;border-radius:10px;transform:scaleX(-1);background:#0b1220}
  #cam-app #cam-overlay{position:absolute;left:0;top:0;width:480px;max-width:100%}
  #cam-app .row{display:flex;gap:14px;flex-wrap:wrap;align-items:flex-start}
  #cam-app canvas#cam-board{border:2px dashed #cbd5e1;border-radius:10px;background:#fff}
  #cam-app .status{background:#f8fafc;border:1px solid #e2e8f0;border-radius:8px;padding:8px;margin-top:8px;font-size:13px}
</style>
<div id="cam-app">
  <div class="card">
    <h3>📷 Camera Game Mode</h3>
    <div>
      <button class="primary" id="cam-start">Bật camera</button>
      <button class="danger" id="cam-stop" disabled>Tắt camera</button>
      <button class="ghost" id="cam-clear">Xóa nét</button>
      <button class="ghost" id="cam-predict">Nhận diện</button>
    </div>
    <div class="row" style="margin-top:10px">
      <div class="stage">
        <video id="cam-video" autoplay playsinline muted></video>
        <canvas id="cam-overlay" width="480" height="360"></canvas>
      </div>
      <div>
        <div style="font-size:12px;color:#64748b;margin-bottom:4px">Nét đã vẽ (đưa vào model)</div>
        <canvas id="cam-board" width="280" height="280"></canvas>
      </div>
    </div>
    <div class="status" id="cam-status">Bấm "Bật camera". Giơ ngón trỏ để vẽ, xòe cả bàn tay để xóa.</div>
    <div class="status" id="cam-result">Chưa nhận diện.</div>
  </div>
</div>
<script>
(function(){
  const $=id=>document.getElementById(id);
  const video=$("cam-video"), overlay=$("cam-overlay"), octx=overlay.getContext("2d");
  const board=$("cam-board"), bctx=board.getContext("2d");
  let stream=null, hands=null, running=false, hasDrawn=false, last=null, lastClear=0, busy=false;

  function clearBoard(){ bctx.fillStyle="#fff"; bctx.fillRect(0,0,280,280);
    bctx.lineCap="round"; bctx.lineJoin="round"; bctx.strokeStyle="#000"; bctx.lineWidth=14; hasDrawn=false; last=null; }
  clearBoard();
  function setStatus(t){ $("cam-status").textContent=t; }

  function loadScript(src){ return new Promise((res,rej)=>{ const s=document.createElement("script");
    s.src=src; s.crossOrigin="anonymous"; s.onload=()=>res(); s.onerror=()=>rej(new Error("Không tải được "+src)); document.head.appendChild(s); }); }

  async function ensureHands(){
    if(hands) return hands;
    if(!window.Hands){ await loadScript("https://cdn.jsdelivr.net/npm/@mediapipe/hands/hands.js"); }
    hands=new window.Hands({locateFile:f=>"https://cdn.jsdelivr.net/npm/@mediapipe/hands/"+f});
    hands.setOptions({maxNumHands:1,modelComplexity:1,minDetectionConfidence:0.7,minTrackingConfidence:0.5});
    hands.onResults(onResults);
    return hands;
  }
  const up=(lm,t,p)=>lm[t].y < lm[p].y-0.015;

  function onResults(r){
    octx.clearRect(0,0,overlay.width,overlay.height);
    const lm = r.multiHandLandmarks && r.multiHandLandmarks[0];
    if(!lm){ last=null; setStatus("Đang tìm bàn tay..."); return; }
    const idx=up(lm,8,6), mid=up(lm,12,10), ring=up(lm,16,14), pinky=up(lm,20,18);
    // vẽ dấu đầu ngón trỏ (đã mirror để khớp video)
    const fx=(1-lm[8].x)*overlay.width, fy=lm[8].y*overlay.height;
    octx.beginPath(); octx.arc(fx,fy,8,0,6.28); octx.fillStyle="#22d3ee"; octx.fill();
    if(idx&&mid&&ring&&pinky){ if(Date.now()-lastClear>1000){ clearBoard(); lastClear=Date.now(); } setStatus("Đã xóa nét."); last=null; return; }
    if(!(idx&&!mid)){ last=null; setStatus("Giơ 1 ngón trỏ để vẽ."); return; }
    const x=(1-lm[8].x)*280, y=lm[8].y*280;
    hasDrawn=true;
    if(last){ bctx.beginPath(); bctx.moveTo(last.x,last.y); bctx.lineTo(x,y); bctx.stroke(); }
    last={x,y}; setStatus("Đang vẽ bằng ngón trỏ...");
  }

  async function loop(){
    if(!running) return;
    try{ await hands.send({image:video}); }catch(e){}
    requestAnimationFrame(loop);
  }
  $("cam-start").onclick=async()=>{
    try{
      setStatus("Đang bật camera...");
      stream=await navigator.mediaDevices.getUserMedia({video:{width:480,height:360},audio:false});
      video.srcObject=stream; await video.play();
      await ensureHands(); running=true; loop();
      $("cam-start").disabled=true; $("cam-stop").disabled=false; setStatus("Camera bật. Giơ ngón trỏ để vẽ.");
    }catch(e){ setStatus("Không bật được camera: "+e.message); }
  };
  $("cam-stop").onclick=()=>{
    running=false;
    if(stream){ stream.getTracks().forEach(t=>t.stop()); stream=null; }
    video.srcObject=null; octx.clearRect(0,0,overlay.width,overlay.height);
    $("cam-start").disabled=false; $("cam-stop").disabled=true; setStatus("Đã tắt camera.");
  };
  $("cam-clear").onclick=clearBoard;
  $("cam-predict").onclick=async()=>{
    if(!hasDrawn){ $("cam-result").textContent="Hãy vẽ gì đó bằng ngón trỏ trước."; return; }
    if(busy)return; busy=true; $("cam-result").textContent="Đang nhận diện...";
    try{
      const res=await google.colab.kernel.invokeFunction("notebook.predict",[board.toDataURL("image/png"),""],{});
      const r=res.data["application/json"];
      $("cam-result").innerHTML="Kết quả: <b>"+r.label+"</b> ("+r.meaning_vi+") · "+r.confidence_percent+"% — "+
        r.top3.map(t=>t.label+" "+(t.confidence*100).toFixed(0)+"%").join(", ");
    }catch(e){ $("cam-result").textContent="Lỗi nhận diện: "+e; }
    finally{ busy=false; }
  };
})();
</script>
"""
display(HTML(CAM_UI))

## 14. Thử nghiệm & so sánh nhiều kiến trúc 🔬

Train nhiều kiến trúc trên cùng tập dữ liệu rồi **so sánh** accuracy, top-3, số tham số và thời gian train:

- **mlp** — baseline chỉ gồm các lớp Dense (để thấy CNN tốt hơn hẳn).
- **cnn_small** — CNN nhỏ, nhanh.
- **cnn_clean** — CNN VGG-style (kiến trúc đang dùng để deploy).
- **cnn_deep** — CNN sâu/rộng hơn, thường chính xác nhất.

Tất cả **không dùng BatchNormalization** (tránh lỗi val/inference ~ngẫu nhiên trên TF 2.21/Keras 3) → ổn định. Dùng tập con để so sánh cho nhanh; chỉnh `CMP_TRAIN_PER_CLASS` / `CMP_EPOCHS` nếu muốn.

In [ ]:
# ---- Cấu hình so sánh (chỉnh cho nhanh/chậm) ----
CMP_TRAIN_PER_CLASS = 2500   # số mẫu train mỗi lớp khi so sánh
CMP_EPOCHS = 14
CMP_PATIENCE = 3

def build_mlp(dropout=0.4):
    inp = keras.Input((28, 28, 1)); x = layers.Flatten()(inp)
    x = layers.Dense(512, activation="relu")(x); x = layers.Dropout(dropout)(x)
    x = layers.Dense(256, activation="relu")(x); x = layers.Dropout(dropout)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)
    return keras.Model(inp, out, name="mlp")

def build_cnn_small(dropout=0.3):
    inp = keras.Input((28, 28, 1))
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inp); x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x); x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x); x = layers.Dropout(dropout)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)
    return keras.Model(inp, out, name="cnn_small")

def build_cnn_clean(dropout=0.4):
    return build_model(dropout)   # kiến trúc đã định nghĩa ở phần 5

def build_cnn_deep(dropout=0.4):
    inp = keras.Input((28, 28, 1)); x = inp
    for f in (48, 96, 160):
        x = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x); x = layers.Dropout(0.25)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(384, activation="relu")(x); x = layers.Dropout(dropout)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)
    return keras.Model(inp, out, name="cnn_deep")

BUILDERS = {"mlp": build_mlp, "cnn_small": build_cnn_small,
            "cnn_clean": build_cnn_clean, "cnn_deep": build_cnn_deep}

# tập con train để so sánh nhanh (lấy từ x_tr/y_tr đã nạp ở phần 4)
_sub = np.concatenate([np.where(y_tr == c)[0][:CMP_TRAIN_PER_CLASS] for c in range(NUM_CLASSES)])
xs, ys = x_tr[_sub], y_tr[_sub]
print(f"Tập so sánh: train={len(xs)}  val={len(x_va)}  test={len(x_te)}")

def train_eval(name, builder):
    import time
    m = builder()
    m.compile(optimizer=keras.optimizers.Adam(LR), loss="categorical_crossentropy", metrics=["accuracy"])
    cb = [keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max",
                                        patience=CMP_PATIENCE, restore_best_weights=True),
          keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)]
    t0 = time.time()
    h = m.fit(make_ds(xs, ys, BATCH_SIZE, True),
              validation_data=make_ds(x_va, y_va, BATCH_SIZE, False),
              epochs=CMP_EPOCHS, callbacks=cb, verbose=0)
    secs = time.time() - t0
    prob = m.predict(make_ds(x_te, y_te, BATCH_SIZE, False), verbose=0)
    pred = prob.argmax(1)
    acc = float((pred == y_te).mean())
    top3 = float(np.mean([t in row for t, row in zip(y_te, np.argsort(prob, 1)[:, -3:])]))
    return {"model": name, "accuracy": acc, "top3": top3,
            "params": int(m.count_params()), "epochs_run": len(h.history["loss"]),
            "train_sec": round(secs, 1)}, m

results, trained = [], {}
for name, b in BUILDERS.items():
    print(f"-> Training {name} ...")
    r, m = train_eval(name, b)
    results.append(r); trained[name] = m
    print(f"   {name}: acc={r['accuracy']*100:.2f}%  top3={r['top3']*100:.2f}%  "
          f"params={r['params']:,}  epochs={r['epochs_run']}  {r['train_sec']}s")
print("\nXong so sánh.")

### Bảng & biểu đồ so sánh

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(results).sort_values("accuracy", ascending=False).reset_index(drop=True)
df_show = df.copy()
df_show["accuracy"] = (df_show["accuracy"] * 100).round(2)
df_show["top3"] = (df_show["top3"] * 100).round(2)
df_show = df_show.rename(columns={"accuracy": "accuracy(%)", "top3": "top3(%)", "train_sec": "train(s)"})
display(df_show)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(df["model"], df["accuracy"] * 100, color="#2563eb")
ax[0].set_title("Accuracy (%) trên test"); ax[0].set_ylim(0, 100)
for i, v in enumerate(df["accuracy"] * 100):
    ax[0].text(i, v + 1, f"{v:.1f}", ha="center", fontsize=9)
ax[1].bar(df["model"], df["params"], color="#16a34a")
ax[1].set_title("Số tham số"); ax[1].ticklabel_format(axis="y", style="plain")
for a in ax:
    a.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

best_name = df.iloc[0]["model"]
print(f"Kiến trúc tốt nhất theo accuracy: {best_name} ({df.iloc[0]['accuracy']*100:.2f}%)")

### (Tùy chọn) Dùng kiến trúc tốt nhất làm model chính

Chạy cell dưới để **thay model đang deploy** bằng kiến trúc thắng cuộc (và lưu lại). Sau đó chạy lại phần 12/13 để demo với model mới. Lưu ý: model so sánh chỉ train trên tập con nhỏ; muốn tốt nhất hãy train lại kiến trúc đó với đầy đủ dữ liệu ở phần 6.

In [ ]:
model = trained[best_name]
model.save(MODEL_PATH)
CATEGORIES_PATH.write_text(json.dumps(CATEGORIES, ensure_ascii=False), encoding="utf-8")
print(f"Đã đặt '{best_name}' làm model chính và lưu vào {MODEL_PATH}")
print("Chạy lại phần 12 hoặc 13 để demo với model mới.")